In [ ]:
!pip install -qq datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 42.8 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
pylibcudf-cu12 25.2.2 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 22.0.0 which is incompatible.
cudf-cu12 25.2.2 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 22.0.0 which is incompatible.
bigframes 2.12.0 requires rich<14,>=12.4.4, but you have rich 14.2.0 which is incompatible.
cudf-polars-cu12 25.6.0 requires pylibcudf-cu12==25.6.*, but you have pylibcudf-cu12 25.2.2 which is incompatible.


In [ ]:
def read_conll(file_path):
    sentences = []
    sentence_labels = []
    unique_labels = set()  # To collect unique labels

    with open(file_path, 'r') as file:
        current_sentence_tokens = []
        current_sentence_labels = []

        for line in file:
            line = line.strip()  # Remove leading/trailing whitespace, including '\n'

            # If it's an empty line, sentence boundary detected
            if not line:
                if current_sentence_tokens:  # Check if there's a sentence to append
                    sentences.append(' '.join(current_sentence_tokens))
                    sentence_labels.append(' '.join(current_sentence_labels))
                current_sentence_tokens = []  # Reset for the next sentence
                current_sentence_labels = []  # Reset for the next sentence
            else:
                line_parts = line.split()  # Split line into token and label
                current_sentence_tokens.append(line_parts[0])

                if len(line_parts) >= 2:
                    current_sentence_labels.append(line_parts[1])
                    unique_labels.add(line_parts[1])  # Add label to the set of unique labels
                else:
                    current_sentence_labels.append('O')  # Default to 'O' if no label provided

    # Append the last sentence if the file doesn't end with an empty line
    if current_sentence_tokens:
        sentences.append(' '.join(current_sentence_tokens))
        sentence_labels.append(' '.join(current_sentence_labels))

    print(f"Unique labels found: {unique_labels}")
    return sentences, sentence_labels

# Load the datasets
test_sentences, test_labels = read_conll('/kaggle/input/nerdata11122/test.txt')
dev_sentences, dev_labels = read_conll('/kaggle/input/nerdata11122/dev.txt')
train_sentences, train_labels = read_conll('/kaggle/input/nerdata11122/train.txt')

# Now, test_sentences, test_labels, dev_sentences, dev_labels, train_sentences, and train_labels are arrays of strings


Unique labels found: {'B-loại_giao_dịch', 'B-bank', 'I-bank', 'B-Tên_tài_khoản', 'I-số_tiền', 'B-số_tiền', 'I-loại_giao_dịch', 'O', 'I-Tên_tài_khoản', 'I-Số_tài_khoản', 'B-Số_tài_khoản', 'I-Loại_tài_khoản', 'B-Loại_tiền_tệ', 'B-giờ_giao_dịch', 'I-Ngày_giao_dịch', 'B-Loại_tài_khoản', 'I-giờ_giao_dịch', 'I-nội_dung_chuyển_tiền', 'B-Ngày_giao_dịch', 'B-nội_dung_chuyển_tiền'}
Unique labels found: {'B-loại_giao_dịch', 'B-bank', 'I-bank', 'B-Tên_tài_khoản', 'I-số_tiền', 'B-số_tiền', 'I-loại_giao_dịch', 'O', 'I-Tên_tài_khoản', 'I-Số_tài_khoản', 'B-Số_tài_khoản', 'I-Loại_tài_khoản', 'B-Loại_tiền_tệ', 'B-giờ_giao_dịch', 'I-Ngày_giao_dịch', 'B-Loại_tài_khoản', 'I-giờ_giao_dịch', 'I-nội_dung_chuyển_tiền', 'B-Ngày_giao_dịch', 'B-nội_dung_chuyển_tiền'}
Unique labels found: {'B-loại_giao_dịch', 'B-bank', 'I-bank', 'B-Tên_tài_khoản', 'I-số_tiền', 'B-số_tiền', 'I-loại_giao_dịch', 'O', 'I-Tên_tài_khoản', 'I-Số_tài_khoản', 'B-Số_tài_khoản', 'I-Loại_tài_khoản', 'B-Loại_tiền_tệ', 'B-giờ_giao_dịch', 'I-Ngà

In [ ]:
test_sentences[1]

'Cho xin trả tiền 12tr500k JPY từ TK thanh toán tới anz Phạm thị thu thủy số điện thoại 0981548789 qua chuyển napas nội dung "chuyển tiết kiệm ngắn hạn" nhé?'

In [ ]:
test_labels[1]

'O O O O B-số_tiền B-Loại_tiền_tệ O B-Loại_tài_khoản I-Loại_tài_khoản I-Loại_tài_khoản O B-bank B-Tên_tài_khoản I-Tên_tài_khoản I-Tên_tài_khoản I-Tên_tài_khoản B-Số_tài_khoản I-Số_tài_khoản I-Số_tài_khoản I-Số_tài_khoản B-loại_giao_dịch I-loại_giao_dịch I-loại_giao_dịch O O B-nội_dung_chuyển_tiền I-nội_dung_chuyển_tiền I-nội_dung_chuyển_tiền I-nội_dung_chuyển_tiền I-nội_dung_chuyển_tiền O'

In [ ]:
from datasets import Dataset

# Step 1: Prepare the datasets from sentences and labels
def prepare_dataset(sentences, labels):
    return {'tokens': sentences, 'labels': labels}

train_dataset = prepare_dataset(train_sentences, train_labels)
dev_dataset = prepare_dataset(dev_sentences, dev_labels)
test_dataset = prepare_dataset(test_sentences, test_labels)

# Step 2: Convert strings of tokens and labels into arrays
def process_string_to_array(dataset):
    return {
        'tokens': [sentence.split() for sentence in dataset['tokens']],
        'labels': [label_seq.split() for label_seq in dataset['labels']]
    }

# Step 3: Process the dataset for token and label lists
train_dataset = process_string_to_array(train_dataset)
dev_dataset = process_string_to_array(dev_dataset)
test_dataset = process_string_to_array(test_dataset)

# Step 4: Convert processed datasets into Hugging Face Dataset objects
train_dataset = Dataset.from_dict(train_dataset)
dev_dataset = Dataset.from_dict(dev_dataset)
test_dataset = Dataset.from_dict(test_dataset)

# Print the size of each dataset and a sample for verification
print(f"Train dataset size: {len(train_dataset)}")
print(f"Dev dataset size: {len(dev_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")
print("Train dataset sample:", train_dataset[0])
print("Dev dataset sample:", dev_dataset[0])
print("Test dataset sample:", test_dataset[0])

# Step 5: Define an Example class
class Example:
    def __init__(self, words, slot_labels, guid=None):
        self.words = words
        self.slot_labels = slot_labels
        self.guid = guid

# Step 6: Convert the dataset to Example objects
def convert_to_examples(dataset):
    return [
        Example(words=tokens, slot_labels=labels, guid=i)
        for i, (tokens, labels) in enumerate(zip(dataset['tokens'], dataset['labels']))
    ]

# Convert datasets into Example objects
train_examples = convert_to_examples(train_dataset)
dev_examples = convert_to_examples(dev_dataset)
test_examples = convert_to_examples(test_dataset)


Train dataset size: 39514
Dev dataset size: 8467
Test dataset size: 8471
Train dataset sample: {'tokens': ['Thứ', '6', 'tôi', 'muốn', 'từ', 'sổ', 'tiết', 'kiệm', 'chuyển', 'khoảng', '3', 'triệu', 'VND', 'sang', 'mã', 'tài', 'khoản', '707172737475', '(', 'LienVietPostBank', ')', 'lời', 'nhắn', 'mua', 'hoa', 'tặng', 'mẹ', '.'], 'labels': ['B-Ngày_giao_dịch', 'I-Ngày_giao_dịch', 'O', 'O', 'O', 'B-Loại_tài_khoản', 'I-Loại_tài_khoản', 'I-Loại_tài_khoản', 'O', 'O', 'B-số_tiền', 'I-số_tiền', 'B-Loại_tiền_tệ', 'O', 'B-Số_tài_khoản', 'I-Số_tài_khoản', 'I-Số_tài_khoản', 'I-Số_tài_khoản', 'O', 'B-bank', 'O', 'O', 'O', 'B-nội_dung_chuyển_tiền', 'I-nội_dung_chuyển_tiền', 'I-nội_dung_chuyển_tiền', 'I-nội_dung_chuyển_tiền', 'O']}
Dev dataset sample: {'tokens': ['Tôi', 'chuyển', 'lúa', '2tr', '300', 'đồng', 'qua', 'mã', 'tài', 'khoản', '150151152153', 'abb', 'Trần', 'Thị', 'Thu', 'Hà', 'qua', 'chuyển', 'napas', 'Thứ', '3', '.'], 'labels': ['O', 'O', 'O', 'B-số_tiền', 'I-số_tiền', 'B-Loại_tiền_tệ', 'O'

In [ ]:
import logging
logger = logging.getLogger(__name__)

import copy
import json
import logging
import os

In [ ]:
# def convert_examples_to_features(
#     examples,
#     max_seq_len,
#     tokenizer,
#     pad_label_id=-100,
#     cls_token_segment_id=0,
#     pad_token_segment_id=0,
#     sequence_segment_id=0,
#     mask_padding_with_zero=True,
# ):
#     # Get special tokens from the tokenizer
#     cls_token = tokenizer.cls_token
#     sep_token = tokenizer.sep_token
#     unk_token = tokenizer.unk_token
#     pad_token_id = tokenizer.pad_token_id

#     # List to hold the converted features
#     features = []

#     for example_index, example in enumerate(examples):
#         # Log progress every 5000 examples
#         if example_index % 400 == 0:
#             logger.info(f"Processing example {example_index} of {len(examples)}")

#         # Tokenize each word and align its corresponding label
#         tokens = []
#         label_ids = []

#         for word, label in zip(example.words, example.slot_labels):
#             word_tokens = tokenizer.tokenize(word)

#             # If the word cannot be tokenized, use [UNK] token
#             if not word_tokens:
#                 word_tokens = [unk_token]

#             tokens.extend(word_tokens)

#             # Map string label to integer ID, apply pad_label_id for subword tokens
#             label_id = label_map[label]
#             label_ids.extend([label_id] + [pad_label_id] * (len(word_tokens) - 1))

#         # Handle sequence truncation for [CLS] and [SEP] tokens
#         special_tokens_count = 2
#         if len(tokens) > max_seq_len - special_tokens_count:
#             tokens = tokens[:max_seq_len - special_tokens_count]
#             label_ids = label_ids[:max_seq_len - special_tokens_count]

#         # Add [SEP] token at the end of the sentence
#         tokens.append(sep_token)
#         label_ids.append(pad_label_id)
#         token_type_ids = [sequence_segment_id] * len(tokens)

#         # Add [CLS] token at the start of the sentence
#         tokens = [cls_token] + tokens
#         label_ids = [pad_label_id] + label_ids
#         token_type_ids = [cls_token_segment_id] + token_type_ids

#         # Convert tokens to input IDs
#         input_ids = tokenizer.convert_tokens_to_ids(tokens)

#         # Create attention masks (1 for real tokens, 0 for padding tokens)
#         attention_mask = [1 if mask_padding_with_zero else 0] * len(input_ids)


#         # Pad sequences to the maximum sequence length
#         padding_length = max_seq_len - len(input_ids)
#         input_ids += [pad_token_id] * padding_length
#         attention_mask += [0 if mask_padding_with_zero else 1] * padding_length
#         token_type_ids += [pad_token_segment_id] * padding_length
#         label_ids += [pad_label_id] * padding_length

#         # Create InputFeatures object and append it to the list of features
#         features.append(
#             InputFeatures(
#                 input_ids=input_ids,
#                 attention_mask=attention_mask,
#                 token_type_ids=token_type_ids,
#                 slot_labels_ids=label_ids,
#             )
#         )

#     return features
# ...existing code...
def convert_examples_to_features(
    examples,
    max_seq_len,
    tokenizer,
    pad_label_id=-100,
    cls_token_segment_id=0,
    pad_token_segment_id=0,
    sequence_segment_id=0,
    mask_padding_with_zero=True,
):
    """
    Tokenize by words using tokenizer(..., is_split_into_words=True) and align labels with word_ids().
    Subword tokens get pad_label_id so loss is ignored for them.
    """
    features = []

    # Ensure tokenizer has pad token
    if tokenizer.pad_token is None:
        tokenizer.add_special_tokens({"pad_token": "[PAD]"})

    for example_index, example in enumerate(examples):
        if example_index % 400 == 0:
            logger.info(f"Processing example {example_index} of {len(examples)}")

        # Tokenize the entire word list at once (handles padding/truncation)
        enc = tokenizer(
            example.words,
            is_split_into_words=True,
            add_special_tokens=True,
            truncation=True,
            padding="max_length",
            max_length=max_seq_len,
            return_attention_mask=True,
            return_token_type_ids=True,
        )

        word_ids = enc.word_ids()  # length == max_seq_len
        label_ids = []
        previous_word_idx = None

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(pad_label_id)
            else:
                if word_idx != previous_word_idx:
                    # first token of the word -> use real label id
                    lbl = example.slot_labels[word_idx]
                    label_ids.append(label_map.get(lbl, pad_label_id))
                else:
                    # subsequent subword token -> ignore in loss
                    label_ids.append(pad_label_id)
                previous_word_idx = word_idx

        input_ids = enc["input_ids"]
        attention_mask = enc["attention_mask"]
        token_type_ids = enc.get("token_type_ids", [sequence_segment_id] * len(input_ids))

        # Sanity check lengths
        assert len(input_ids) == max_seq_len
        assert len(attention_mask) == max_seq_len
        assert len(token_type_ids) == max_seq_len
        assert len(label_ids) == max_seq_len

        features.append(
            InputFeatures(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids,
                slot_labels_ids=label_ids,
            )
        )

    return features
# ...existing code...

In [ ]:
# Define the label list (ensure that it includes all labels from your dataset)
label_list = ['O', 'B-nội_dung_chuyển_tiền', 'I-bank', 'I-giờ_giao_dịch', 'I-loại_giao_dịch', 'I-Số_tài_khoản', 'B-Loại_tài_khoản', 'I-nội_dung_chuyển_tiền', 'B-bank', 'I-số_tiền', 'B-giờ_giao_dịch', 'B-Ngày_giao_dịch', 'B-Loại_tiền_tệ', 'B-loại_giao_dịch', 'I-Ngày_giao_dịch', 'B-Tên_tài_khoản', 'I-Loại_tài_khoản', 'I-Tên_tài_khoản', 'B-số_tiền', 'B-Số_tài_khoản']

# Create a mapping from label strings to integers
label_map = {label: i for i, label in enumerate(label_list)}


In [ ]:
import json

In [ ]:
class InputFeatures(object):
    """A single set of features of data."""

    def __init__(self, input_ids, attention_mask, token_type_ids, slot_labels_ids):
        self.input_ids = input_ids
        self.attention_mask = attention_mask
        self.token_type_ids = token_type_ids
        self.slot_labels_ids = slot_labels_ids

    def __repr__(self):
        return str(self.to_json_string())

    def to_dict(self):
        """Serializes this instance to a Python dictionary."""
        output = copy.deepcopy(self.__dict__)
        return output

    def to_json_string(self):
        """Serializes this instance to a JSON string."""
        return json.dumps(self.to_dict(), indent=2, sort_keys=True) + "\n"

In [ ]:
from transformers import AutoTokenizer

# Initialize the tokenizer
tokenizer = AutoTokenizer.from_pretrained('NlpHUST/electra-base-vn')

# Set the maximum sequence length
max_seq_len = 200  # You can adjust this based on your model/input

# Convert examples to features
train_features = convert_examples_to_features(train_examples, max_seq_len, tokenizer)
dev_features = convert_examples_to_features(dev_examples, max_seq_len, tokenizer)
test_features = convert_examples_to_features(test_examples, max_seq_len, tokenizer)


tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

In [ ]:
tokenizer.cls_token, tokenizer.sep_token, tokenizer.unk_token, tokenizer.pad_token_id

('[CLS]', '[SEP]', '[UNK]', 0)

In [ ]:
import torch
from torch.utils.data import Dataset

# Define a Dataset class to wrap the tokenized features for training
class NERDataset(Dataset):
    def __init__(self, features):
        self.features = features

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        feature = self.features[idx]
        return {
            'input_ids': torch.tensor(feature.input_ids, dtype=torch.long),
            'attention_mask': torch.tensor(feature.attention_mask, dtype=torch.long),
            'token_type_ids': torch.tensor(feature.token_type_ids, dtype=torch.long),
            'labels': torch.tensor(feature.slot_labels_ids, dtype=torch.long),
        }

# Convert tokenized features into PyTorch datasets
train_dataset = NERDataset(train_features)
dev_dataset = NERDataset(dev_features)
test_dataset = NERDataset(test_features)


In [ ]:
train_dataset[0]

{'input_ids': tensor([    2,  1055,   459,   161,   475,    41,  1655,   532,  1387,   306,
           324,   220,   248,  3001,   607,  1253,   225,   784, 21441,  4459,
         45531, 30977,  1850,    22, 57848, 36831,  9555,    23,   483,  1899,
           382,   730,   905,   574,     6,     3,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,   

In [ ]:
from transformers import AutoModelForTokenClassification

# Define the number of unique labels (ensure this matches your dataset's label set)
num_labels = len(label_list)  # e.g., the number of unique labels such as O, B-ORG, etc.

# Load the Electra model for token classification
model = AutoModelForTokenClassification.from_pretrained('NlpHUST/electra-base-vn', num_labels=num_labels)


2025-12-11 03:13:43.026022: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765422823.177450      47 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765422823.222538      47 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

model.safetensors:   0%|          | 0.00/535M [00:00<?, ?B/s]

Some weights of ElectraForTokenClassification were not initialized from the model checkpoint at NlpHUST/electra-base-vn and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback

# Define training arguments
training_args = TrainingArguments(
    output_dir='./results',
    eval_strategy="epoch",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    weight_decay=0.01,
    learning_rate=2e-5,  # Thêm learning rate
    warmup_steps=500,    # Thêm warmup
    logging_dir='./logs',
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=3,  # Lưu 3 checkpoints tốt nhất
    load_best_model_at_end=True,  # Load best model
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none",
)

In [ ]:
!pip install seqeval

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=e2a754aa41482a9cbda8eb6eebead1beaec03d264929ae1e48eacb475d94ba01
  Stored in directory: /root/.cache/pip/wheels/bc/92/f0/243288f899c2eacdfa8c5f9aede4c71a9bad0ee26a01dc5ead
Successfully built seqeval


In [ ]:
label_list

['O',
 'B-nội_dung_chuyển_tiền',
 'I-bank',
 'I-giờ_giao_dịch',
 'I-loại_giao_dịch',
 'I-Số_tài_khoản',
 'B-Loại_tài_khoản',
 'I-nội_dung_chuyển_tiền',
 'B-bank',
 'I-số_tiền',
 'B-giờ_giao_dịch',
 'B-Ngày_giao_dịch',
 'B-Loại_tiền_tệ',
 'B-loại_giao_dịch',
 'I-Ngày_giao_dịch',
 'B-Tên_tài_khoản',
 'I-Loại_tài_khoản',
 'I-Tên_tài_khoản',
 'B-số_tiền',
 'B-Số_tài_khoản']

In [ ]:
from transformers import EvalPrediction
def compute_metrics(p: EvalPrediction):
    predictions = p.predictions.argmax(axis=2)  # Get predicted label indices
    labels = p.label_ids  # True label IDs

    # Debugging: Print shapes of predictions and labels
    print(f"Shape of predictions: {predictions.shape}")
    print(f"Shape of labels: {labels.shape}")

    # Debugging: Log first few predictions and labels for inspection
    print(f"First few predictions: {predictions[:2]}")
    print(f"First few labels: {labels[:2]}")

    pred_labels = []
    true_labels = []

    # Iterate through predictions and labels
    for i, (pred_seq, true_seq) in enumerate(zip(predictions, labels)):
        pred_label_seq = []
        true_label_seq = []

        # Iterate through each token in the sequence
        for pred_idx, true_idx in zip(pred_seq, true_seq):
            if true_idx == -100:
                # Debugging: Log any padding tokens encountered
                # print(f"Padding token encountered at position {i}")
                continue

            # Check if the indices are within the valid range
            if pred_idx < len(label_list) and true_idx < len(label_list):
                pred_label_seq.append(label_list[pred_idx])
                true_label_seq.append(label_list[true_idx])
            else:
                # Debugging: Log when out-of-bound indices are encountered
                print(f"Index out of range: pred_idx={pred_idx}, true_idx={true_idx} at position {i}")

        pred_labels.append(pred_label_seq)
        true_labels.append(true_label_seq)

    # Debugging: Log final processed predictions and labels
    print(f"Processed pred_labels: {pred_labels[:2]}")
    print(f"Processed true_labels: {true_labels[:2]}")

    # Compute token-level F1, Precision, and Recall
    precision = precision_score(true_labels, pred_labels)
    # Trong 10 lần dự đoán nhãn X: thì chúng ta đoán đúng 6 lần -> 6/10 = 60%

    recall = recall_score(true_labels, pred_labels)
    # Trong 8 nhãn X thật: thì chúng ta đoán đúng 6 lần -> 6/8 = 75%

    f1 = f1_score(true_labels, pred_labels)

    # Debugging: Print classification report
    print("Classification Report:")
    print(classification_report(true_labels, pred_labels))

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

In [ ]:
from transformers import Trainer
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score
from transformers import EvalPrediction


In [ ]:
# Initialize the Trainer with the modified compute_metrics function
import os
os.environ["WANDB_DISABLED"] = "true"
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,  # Updated function
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

# Train the model
trainer.train()


/tmp/ipykernel_47/3911054465.py:4: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.022700,0.009486,0.995713,0.996694,0.996203
2,0.003000,0.003650,0.998602,0.998951,0.998776
3,0.001300,0.001942,0.999015,0.999174,0.999094
4,0.002200,0.001863,0.999015,0.999301,0.999158
5,0.000400,0.001781,0.999015,0.999396,0.999205
6,0.000400,0.001878,0.999460,0.999364,0.999412
7,0.000200,0.002068,0.999491,0.999428,0.999460
8,0.000200,0.001358,0.999491,0.999491,0.999491
9,0.000100,0.001414,0.999428,0.999523,0.999476
10,0.000100,0.001243,0.999491,0.999587,0.999539


Shape of predictions: (8467, 200)
Shape of labels: (8467, 200)
First few predictions: [[ 0  0  0  0 18  9  9 12  0 19  5  5  5  5  5  5  5  8  2 15 17 17 17 13
   4  4  4 11 14  0  0  0  8 14  8 14 14  0  0  0  0  0  9  9  9  9  0  0
   4  4  0  9 16  9  0  8  5  5  8 17  5  2  8  8  8  2  0  4  2 14 16  4
   4  4  0 14 14  0  0  0  8  0  0 17  0  0  8  4  4  0 14 14  0  0  5  5
   5  5  5  0  0  0  0  0  9  9  9  9  0  0  0  5  5  5  5  5  5  5  5  5
   0 17 17  0  8  8  8  2  0 17 17  8  8  4  4 14  0 14 14  0  0  0  0  0
  17 17  0  0  0  0 18  9  9  9  9  0  8  5  5  5  5  5  5  5  5  0  8  2
   0 17 17 17  8  4  4  4  4 16 14 14  8  8  5  4  5  8  5 14  0  0  0  0
   0  0 17 17 17  9  0  0]
 [ 0  0  0  0  0  0 18  9 12  0  6 16 16 16  0 19  5  5  5  5  5  5 15 17
  17  8  2 11 14 14  0  0  1  7  7  7 13  4  4  0  0  0  0  0  1  9  0  0
   0  7  1  6 13  9  9  0  6 16 16 16 16  0 16 16  8  5 16 16 16  0  0  0
   1  7  7  7 13  4  7  7  0  7  5  7  7  0  7  8  7  0 14 14  0  0  1  7

/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Shape of predictions: (8467, 200)
Shape of labels: (8467, 200)
First few predictions: [[ 0  0  0  0 18  9  9 12  0 19  5  5  5  5  5  5  5  8  2 15 17 17 17 13
   4  4  4 11 14  0  0  8  8 14  8 14 14  8  0  8  0  8  9  9  9  9  8  8
   4  9  8  9  5  9  8  8  5  5  8  5  5  2  8  8  8  2  8  8  2 14  8  4
   4  4  8 11 14  8  8  8  8  2  8 17  8  8  8  4 16  8 14 14  8  8  5  5
   5  5  5  8  0  0  0  8  9  9  9  9  0  0  0  5  5  5  5  5  5  5  5  5
   8 17 17  5  8  8  8  2  8 17 17  2  8  4  4  2  8 14 14  8  8  8  2  8
   9 17  0  0  0  0  9  9  9  9 12  0  8  5  5  5  5  5  5  5  5  8  8  2
   0 17 17 17  8  2  4  4  2 11 14 14  8  8  5  4  8 11  5 14  8  8  8  8
   2  8  2 17 17 12  8  8]
 [ 0  0  0  0  0  0 18  9 12  0  6 16 16 16  0 19  5  5  5  5  5  5 15 17
  17  8  2 11 14 14  0  0  1  7  7  7 13  4  4  0  0  0  0  0  8  9  9  8
   8 16  6 16 13 16  9  8  8 16 16 16 16  8 16 16  8  5  2  2  2  2  8  0
   1  7  7  7 13  4  4  0  8  7  5  5  8  8  0  8  2  2 11 14  0  1  1  7

...

/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Shape of predictions: (8467, 200)
Shape of labels: (8467, 200)
First few predictions: [[ 0  0  0  0 18  9  9 12  0 19  5  5  5  5  5  5  5  8  2 15 17 17 17 13
   4  4  4 11 14  0  0  8 13 14  5  5  5  9  0  0  0  0  9  9  9  9  9  0
   4  9  0  9  9  9  0  8  5  5  0  5  5  2  8  8  8  2  0  0  5 14  0  4
   4  4  0 14 14  5  0  8  8  2  0 17 17  0  0  4  4  0 14 14  0  8  5  5
   5  5  5  9  0  0  0  0  9  9  9  9  0  0  0  5  5  9  5  5  5  5  5  2
   0 17 17 17  8  8  8  2 15 17 17 17  8  4  4  4  0 14 14  0  0  8  2  0
  17 17  9  0  0  0  9  9  9  9  9  0  0  5  5  5  5  5  5  5  5  8  8  2
   0 17 17 17  0 16  4  4  2  0 14  5  0  8  5  4  5  0  5  5  5  0  0  8
   0  0 17 17 17 17  0  0]
 [ 0  0  0  0  0  0 18  9 12  0  6 16 16 16  0 19  5  5  5  5  5  5 15 17
  17  8  2 11 14 14  0  0  1  7  7  7 13  4  4  0  0  0  0  9 13  9  9 16
   8 16  7  6 13  9  9  0 16 16  5 16 16  0 16 16  8  5  2  5 14  0  0  0
   1  7  7  7 13  4  4  0  0  5  5  7  4 16 17  8  2 16 14 14  0  0  1  7

/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Shape of predictions: (8467, 200)
Shape of labels: (8467, 200)
First few predictions: [[ 0  0  0  0 18  9  9 12  0 19  5  5  5  5  5  5  5  8  2 15 17 17 17 13
   4  4  4 11 14  0  0  8 13 14  5  5  5  5  0  0  0  0  9  9  9  9  9  0
   4  9  0  9  9  9  0  8  5  5  0  5  5  2  8  8  8  2  0  0 14 14  0  4
   4  4  0 14 14  5  0  8  8  2  0 17 17  0  0  4  4  0 14 14  0  0  5  5
   5  5  5  9  0  0  0  0  9  9  9  9  0  0  0  5  5  9  5  5  5  5  5  2
   0 17 17 17  8  8  8  2 15 17 17 17  8  4  4  4  0 14 14  0  0  0  2  0
  17 17  9  0  0  0  9  9  9  9  9  0  0  5  5  5  5  5  5  5  5  0  8  2
   0 17 17 17  0 16  4  4  2  0 14  5  0  8  5  4  5  0  5  5  5  0  0  8
   0  0 17 17 17 17  0  0]
 [ 0  0  0  0  0  0 18  9 12  0  6 16 16 16  0 19  5  5  5  5  5  5 15 17
  17  8  2 11 14 14  0  0  1  7  7  7 13  4  4  0  0  0  0  9 13  9  9 16
   8 16  7  0 13  9  9  0 16 16 16 16 16  0 16 16  8  5  2  2 14  0  0  0
   1  7  7  7 13  4  4  0  0  5  5  7  4 16 17  8  2 16 14 14  0  0  1  7

TrainOutput(global_step=12350, training_loss=0.04910340451585932, metrics={'train_runtime': 9228.1243, 'train_samples_per_second': 42.819, 'train_steps_per_second': 1.338, 'total_flos': 4.033812611184e+16, 'train_loss': 0.04910340451585932, 'epoch': 10.0})

In [ ]:
# Lưu model tốt nhất
best_model_path = "./best_model"

# Lưu model
trainer.save_model(best_model_path)

# Lưu tokenizer
tokenizer.save_pretrained(best_model_path)

# Lưu label mapping để sử dụng sau
import json
with open(f"{best_model_path}/label_map.json", "w") as f:
    json.dump(label_map, f, ensure_ascii=False, indent=2)

print(f"Model đã được lưu tại: {best_model_path}")

Model đã được lưu tại: ./best_model


In [ ]:
# Load model best đã lưu
from transformers import AutoModelForTokenClassification

best_model = AutoModelForTokenClassification.from_pretrained(best_model_path)

# Tạo trainer mới với model best
from transformers import Trainer

best_trainer = Trainer(
    model=best_model,
    args=training_args,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

# Đánh giá trên tập test với model best
print("Đang đánh giá model best trên tập test...")
test_results = best_trainer.evaluate(test_dataset)

print("\n=== KẾT QUẢ MODEL BEST TRÊN TẬP TEST ===")
print(f"Test Loss: {test_results['eval_loss']:.4f}")
print(f"Test Precision: {test_results['eval_precision']:.4f}")
print(f"Test Recall: {test_results['eval_recall']:.4f}")
print(f"Test F1: {test_results['eval_f1']:.4f}")

/tmp/ipykernel_47/1057801101.py:9: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  best_trainer = Trainer(


Đang đánh giá model best trên tập test...


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Shape of predictions: (8471, 200)
Shape of labels: (8471, 200)
First few predictions: [[ 0  0  0  0  0 18  9 12  0  8  2 15 17 17 19  5  5  5  5 10  3  3 13  4
   4  4  4 11 14 14 14 14  0  0  8 17 17  0  0 14  0  0 13  9  9  9  9  8
   0  0  0  9  9  9  8  8  2  2  2 17  2 13  5  5  5  5  0 14 14 11  4 14
  14  8  8 14 17 11 16 14  3  5 10  3 13  4 13  0  0 11 14 14 14 14 14  0
  17 17 17  0  0  0  0  0  0  9  9  9  9  0  0  0  0 17 17  0  8  8  5  5
  10  5  2 13  5  5  5  5 10  3  5 13  4  5 16  8 11 14 14 14 14  5  5 10
   9  9  0  0  0  0  0  0  9  9  9  0  8  2  0 17 17 17  0  0  5  5  5  5
  10  5  5 13  4  4  4  5  0 11 14 14 14 14  8  0  0 17 17  0 11 14 14  5
   5  0  9  9  9  0  0  0]
 [ 0  0  0  0  0 18  9  9  9 12  0  6 16 16  0  8  2 15 17 17 17 19  5  5
   5  5  5  5 13  4  4  4  0  0  1  1  7  7  7  7  7  0  0  0  0  9  9  0
   0  9  9  9  9  9  9  9 16  0 16 17 16  0  8  2 15  2  5  5  5  4  4  4
   0  0  1  1  7  7  7  7  7  1  0  7  7 16  5  0  5 12  9  4  4  4  0  0

In [ ]:
import json
import re
import torch
from transformers import AutoModelForTokenClassification, AutoTokenizer
from typing import Optional, List, Tuple

class NERPipeline:
    def __init__(self, model_path, device: Optional[torch.device] = None) -> None:
        self.model = AutoModelForTokenClassification.from_pretrained(model_path)
        self.tokenizer = AutoTokenizer.from_pretrained(model_path)

        with open(f"{model_path}/label_map.json", "r", encoding="utf-8") as f:
            self.id_to_label = {v: k for k, v in json.load(f).items()}

        self.device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device).eval()

        self.whitespace_pattern = re.compile(r'\s+')
        self.time_pattern = re.compile(r'(\d+)\s*:\s*(\d+)')
        self.decimal_pattern = re.compile(r'(\d+)\s*\.\s*(\d+)')
        self.comma_pattern = re.compile(r'(\d+)\s*,\s*(\d+)')
        self.digit_pattern = re.compile(r'\d+')
        self.digit_search = re.compile(r'\d')
        self.punct_pattern = re.compile(r'^[.,;!?:"\'()[\]{}«»`~*—–\-\s]+|[.,;!?:"\'()[\]{}«»`~*—–\-\s]+$')
        self.special_only = re.compile(r'[^\w\s]+')

    def postprocess_entity(self, entity_tokens: List[str], entity_type: str) -> str:
        text = " ".join(entity_tokens).replace("_", " ")
        text = self.whitespace_pattern.sub(' ', text).strip()

        text = self.time_pattern.sub(r'\1:\2', text)
        text = self.decimal_pattern.sub(r'\1.\2', text)
        text = self.comma_pattern.sub(r'\1,\2', text)

        if entity_type == "Số_tài_khoản":
            nums = self.digit_pattern.findall(text)
            text = ''.join(nums) if nums else text
        elif entity_type == "giờ_giao_dịch":
            m = self.digit_search.search(text)
            text = text[m.start():].strip() if m else text

        text = self.punct_pattern.sub('', text)

        return "" if self.special_only.fullmatch(text) else text

    def predict(self, text: str, max_len: int = 200) -> List[Tuple[str, str]]:
        text = text.replace('(', '').replace(')', '')
        words = text.split()

        encoded = self.tokenizer(words, is_split_into_words=True,
                                 return_tensors="pt",
                                 padding=True, truncation=True,
                                 max_length=max_len)
        word_ids = encoded.word_ids()
        inputs = {k: v.to(self.device) for k, v in encoded.items()}

        with torch.no_grad():
            preds = torch.argmax(self.model(**inputs).logits, dim=2).cpu().numpy()[0]

        entities, cur_type, cur_tokens = [], None, []
        prev_idx = None
        for idx, word_idx in enumerate(word_ids):
            if word_idx is not None and word_idx != prev_idx:
                label = self.id_to_label.get(preds[idx], "O")
                if label.startswith("B-"):
                    if cur_type:
                        entities.append((cur_type, cur_tokens[:]))
                    cur_type = label[2:]
                    cur_tokens = [words[word_idx]]
                elif label.startswith("I-") and cur_type == label[2:]:
                    cur_tokens.append(words[word_idx])
                else:
                    if cur_type:
                        entities.append((cur_type, cur_tokens[:]))
                        cur_type, cur_tokens = None, []
                prev_idx = word_idx
        if cur_type:
            entities.append((cur_type, cur_tokens))

        result = []
        for etype, tokens in entities:
            if tokens:
                processed_value = self.postprocess_entity(tokens, etype)
                if processed_value:
                    result.append((etype, processed_value))
        return result

    def print_iob_tags(self, text : str, max_len : int = 200) -> None:
        words = text.replace('(', '').replace(')', '').split()
        encoded = self.tokenizer(words, is_split_into_words=True,
                                 return_tensors="pt",
                                 padding=True, truncation=True,
                                 max_length=max_len)
        word_ids = encoded.word_ids()
        inputs = {k: v.to(self.device) for k, v in encoded.items()}

        with torch.no_grad():
            preds = torch.argmax(self.model(**inputs).logits, dim=2).cpu().numpy()[0]

        prev_idx = None
        for idx, word_idx in enumerate(word_ids):
            if word_idx is not None and word_idx != prev_idx:
                label = self.id_to_label.get(preds[idx], "O")
                print(f"{words[word_idx]:<20} {label}")
                prev_idx = word_idx

# Chạy với các câu
ner = NERPipeline("best_model")
text = 'Xem cac giao dich chuyen tien cua stk 2323423432 voi ghi chu chuyen tien cho ban'

print(f"Câu gốc: {text}\n")

# # IOB tags
# print("IOB Tags:")
# ner.print_iob_tags(text)

# Entity extraction
print("\nEntities:")
for etype, value in ner.predict(text):
    print(f"{etype:<30} {value}")

Câu gốc: Xem cac giao dich chuyen tien cua stk 2323423432 voi ghi chu chuyen tien cho ban


Entities:
loại_giao_dịch                 chuyen tien
Số_tài_khoản                   2323423432
nội_dung_chuyển_tiền           chuyen tien cho ban


In [ ]:
!cd /kaggle/working && zip -r best_model.zip best_model


  adding: best_model/ (stored 0%)
  adding: best_model/tokenizer.json (deflated 69%)
  adding: best_model/label_map.json (deflated 61%)
  adding: best_model/training_args.bin (deflated 52%)
  adding: best_model/special_tokens_map.json (deflated 42%)
  adding: best_model/model.safetensors

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


 (deflated 7%)
  adding: best_model/config.json (deflated 64%)
  adding: best_model/tokenizer_config.json (deflated 74%)
  adding: best_model/vocab.txt (deflated 43%)
